In [2]:
"""
Driving Stress Calculation
Author: Hannah Picton 
Date: 27th March 2026
Description: Using Equation 3 of the associated manuscript, this script calculates the relavant contribution of changes in ice thickness (H) and ice surface slope (S) to the increase in driving stress, 
and consequent ice flow acceleration, observed at LEV6 between 2009 and 2021. A conservative estimate of uncertainty in the height component (±0.5 m) is also propogated. 
"""

# Import libraries 
import math
import numpy as np

In [ ]:
# ==========================================================================================
# Calculate acceleration due to increased driving stress, with no uncertainty applied 
# ==========================================================================================

# Define values for LEV5
lev5_2009_height = 1442.92
lev5_2009_bed = 243.05
lev5_2009_thickness = lev5_2009_height - lev5_2009_bed
lev5_2021_height = 1438.01
lev5_2021_bed = 243.05
lev_5_2021_thickness = lev5_2021_height - lev5_2021_bed

# Define values for LEV6
lev6_2009_height = 1672.55
lev6_2009_bed = 234.31
lev6_2009_thickness = lev6_2009_height - lev6_2009_bed
lev6_2021_height = 1670.83
lev6_2021_bed = 234.31
lev6_2021_thickness = lev6_2021_height - lev6_2021_bed

# Distance between LEV5 and LEV6
dist_2009 = 35961.214
dist_2021 = 35985.827

# Calculate slope between sites
slope_2009 = (lev6_2009_height - lev5_2009_height) / dist_2009
slope_2021 = (lev6_2021_height - lev5_2021_height) / dist_2021

# Calculate changes in slope and ice thickness
delta_s = slope_2021 - slope_2009
delta_thickness = lev6_2021_thickness - lev6_2009_thickness

# Calculate changes in acceleration due to changes in driving stress
acceleration = 62.6 * (3 * (delta_s / slope_2009) + 3 * (delta_thickness / lev6_2009_thickness))
print(f"Acceleration due to increased driving stress: + {acceleration:.3f} m")

# Calculate percentage contribution to observed velocity increase (+3.2 m)
observed_delta_v = 3.2
percentage_of_observed = (acceleration / observed_delta_v) * 100
print(f"Percentage contribution to observed velocity increase: {percentage_of_observed:.2f} %")

Acceleration due to increased driving stress: + 2.254 m
Percentage contribution to observed velocity increase: 70.44 %


In [ ]:
# ==========================================================================================
# Incorporate uncertainty in height (±0.5 m) and calculate range of possible accelerations
# ==========================================================================================

uncertainty_H = 0.5       

# Nominal velocity change (i.e. no uncertainty)
A = 3.0
B = 3.0
C = 62.6
X_S = delta_s / slope_2009
X_T = delta_thickness / lev6_2009_thickness
delta_v_nominal = C * (A * X_S + B * X_T)
nom_slope = C * A * X_S # Nominal slope component
nom_thick = C * B * X_T # Nominal thickness component

# Calculate uncertainty in slope 
uncertainty_H_diff = math.sqrt(uncertainty_H**2 + uncertainty_H**2)
delta_slope_2009 = abs(slope_2009) * (uncertainty_H_diff / (lev6_2009_height - lev5_2009_height))
delta_slope_2021 = abs(slope_2021) * (uncertainty_H_diff / (lev6_2021_height - lev5_2021_height))
delta_delta_s = math.sqrt(delta_slope_2009**2 + delta_slope_2021**2)

# Calculate uncertainty in thickness change
delta_T = math.sqrt(uncertainty_H**2 + uncertainty_H**2)
delta_delta_T = math.sqrt(delta_T**2 + delta_T**2)

# Calculate uncertainty in fractional terms (relative changes in slope and thickness)
fractional_uncertainty_slope = math.sqrt((delta_delta_s / abs(delta_s))**2 + (delta_slope_2009 / abs(slope_2009))**2)
delta_X_S = abs(X_S) * fractional_uncertainty_slope
fractional_uncertainty_thickness = math.sqrt((delta_delta_T / abs(delta_thickness))**2 + (uncertainty_H / lev6_2009_thickness)**2)
delta_X_T = abs(X_T) * fractional_uncertainty_thickness

# Convert uncertainty in fractional terms to absolute uncertainty in velocity components
unc_slope = C * A * delta_X_S
unc_thick = C * B * delta_X_T
unc_total = C * math.sqrt((A * delta_X_S)**2 + (B * delta_X_T)**2)

In [5]:
# ==========================================================================================
# Print results in a clear and concise format, showing both absolute and relative contributions, as well as the back-calculated slope angle envelope for 2009 and 2021. 
# ==========================================================================================

print(f"Observed Change in Velocity (LEV6; 2009 - 2021): +{observed_delta_v:.1f} m\n")

print("--- Absolute Values (m) ---")
print(f"{'Component':<32} | {'Nominal':>10} | {'Uncertainty (+/-)':>18} | {'Range (Min to Max)':>22}")
print("-" * 86)

def print_comp(label, nom, unc):
    print(f"{label:<32} | {nom:>10.3f} | {unc:>18.3f} | {nom-unc:>10.3f} to {nom+unc:<10.3f}")

print_comp("Total Model (delta u_s)", delta_v_nominal, unc_total)
print_comp("Slope (3 * deltaS/S)", nom_slope, unc_slope)
print_comp("Thickness (3 * deltaH/H)", nom_thick, unc_thick)

print("\n")
print(f"--- Relative Values (% of observed + {observed_delta_v:.1f} m) ---")
print(f"{'Component':<32} | {'Nominal %':>10} | {'Uncertainty % (+/-)':>18} | {'Range % (Min to Max)':>22}")
print("-" * 89)

def print_pct(label, nom, unc):
    nom_pct = (nom / observed_delta_v) * 100
    unc_pct = (unc / observed_delta_v) * 100
    print(f"{label:<32} | {nom_pct:>9.1f}% | {unc_pct:>17.1f}% | {nom_pct-unc_pct:>9.1f}% to {nom_pct+unc_pct:<10.1f}%")

print_pct("Total Model (delta u_s)", delta_v_nominal, unc_total)
print_pct("Slope (3 * deltaS/S)", nom_slope, unc_slope)
print_pct("Thickness  (3 * deltaH/H)", nom_thick, unc_thick)
print("\n")


print("--- Back-calculate Slope Angle Envelope ---")
print("-" * 89)

def print_angle(label, slope, delta_slope):
    angle = math.degrees(math.atan(slope))
    delta_angle = (1/(1+slope**2)) * delta_slope * (180/math.pi)
    print(f"{label} Slope Angle:")
    print(f"  Nominal:     {angle:.6f} degrees")
    print(f"  Uncertainty: +/- {delta_angle:.6f} degrees")
    print(f"  Range:       ({angle - delta_angle:.6f} to {angle + delta_angle:.6f}) degrees")
    print(f"  (Slope ratio: {slope:.6f} +/- {delta_slope:.6f})\n")

print_angle("2009", slope_2009, delta_slope_2009)
print_angle("2021", slope_2021, delta_slope_2021)
# ==========================================================================================

Observed Change in Velocity (LEV6; 2009 - 2021): +3.2 m

--- Absolute Values (m) ---
Component                        |    Nominal |  Uncertainty (+/-) |     Range (Min to Max)
--------------------------------------------------------------------------------------
Total Model (delta u_s)          |      2.254 |              0.828 |      1.426 to 3.082     
Slope (3 * deltaS/S)             |      2.479 |              0.818 |      1.661 to 3.296     
Thickness (3 * deltaH/H)         |     -0.225 |              0.131 |     -0.355 to -0.094    


--- Relative Values (% of observed + 3.2 m) ---
Component                        |  Nominal % | Uncertainty % (+/-) |   Range % (Min to Max)
-----------------------------------------------------------------------------------------
Total Model (delta u_s)          |      70.4% |              25.9% |      44.6% to 96.3      %
Slope (3 * deltaS/S)             |      77.5% |              25.5% |      51.9% to 103.0     %
Thickness  (3 * deltaH/H)      